# FinSight — RAG Pipeline Walkthrough

This notebook walks through the full pipeline step by step:
1. Document ingestion and chunking
2. Embedding and vector storage
3. Semantic retrieval
4. Agentic Q&A with source attribution

Use the sample document in `data/sample_docs/` or drop in any financial PDF.

In [ ]:
import os
import sys
sys.path.append('..')

from dotenv import load_dotenv
load_dotenv('../.env')

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
assert OPENAI_API_KEY, 'Set OPENAI_API_KEY in your .env file'

## 1. Document Ingestion

The `DocumentProcessor` loads a file, cleans the text, and splits it into overlapping chunks.
Chunk size and overlap are configurable — larger chunks preserve more context per retrieval;
smaller chunks improve precision.

In [ ]:
from src.ingestion import DocumentProcessor

processor = DocumentProcessor(chunk_size=512, chunk_overlap=50)
chunks = processor.process('../data/sample_docs/acme_q3_2024_earnings.txt', source_name='acme_q3_2024.txt')

print(f'Total chunks: {len(chunks)}')
print(f'\nFirst chunk preview:')
print('-' * 60)
print(chunks[0]['content'][:400])

In [ ]:
# Inspect chunk length distribution
import matplotlib.pyplot as plt

lengths = [len(c['content'].split()) for c in chunks]

plt.figure(figsize=(8, 3))
plt.hist(lengths, bins=15, color='steelblue', edgecolor='white')
plt.xlabel('Chunk length (words)')
plt.ylabel('Count')
plt.title('Chunk Length Distribution')
plt.tight_layout()
plt.show()

print(f'Min: {min(lengths)} | Max: {max(lengths)} | Avg: {sum(lengths)/len(lengths):.0f} words')

## 2. Embedding and Vector Storage

Chunks are embedded using OpenAI's `text-embedding-3-small` (1536 dimensions) and stored in ChromaDB.
ChromaDB uses HNSW under the hood for approximate nearest neighbor search.

In [ ]:
from src.vector_store import VectorStore

vs = VectorStore(api_key=OPENAI_API_KEY)
vs.add_documents(chunks)

print(f'Documents in store: {vs.count()}')

## 3. Semantic Retrieval

Given a query, we embed it and retrieve the top-k most similar chunks by cosine similarity.
Notice how the retrieval is query-aware — different questions pull different chunks even from the same document.

In [ ]:
query = 'What was the revenue growth and EPS this quarter?'
results = vs.search(query, top_k=3)

for i, r in enumerate(results):
    print(f'--- Result {i+1} | Score: {r["score"]:.3f} | Source: {r["source"]} | Chunk {r["chunk_id"]} ---')
    print(r['content'][:300])
    print()

In [ ]:
# Try a different query to see different chunks retrieved
query2 = 'What are the main risk factors mentioned?'
results2 = vs.search(query2, top_k=3)

for i, r in enumerate(results2):
    print(f'--- Result {i+1} | Score: {r["score"]:.3f} | Chunk {r["chunk_id"]} ---')
    print(r['content'][:300])
    print()

## 4. Agentic Q&A

The `FinancialAgent` wraps the retrieval and generation steps in a ReAct-style loop.
It uses OpenAI's function-calling API to decide *when* to retrieve and *what to search for*,
allowing it to decompose multi-part questions into multiple retrieval steps.

In [ ]:
from src.agent import FinancialAgent

agent = FinancialAgent(vector_store=vs, api_key=OPENAI_API_KEY)

result = agent.run('What was the revenue growth YoY and what guidance did management provide for Q4?')

print('ANSWER')
print('=' * 60)
print(result['answer'])

print('\nSOURCES')
print('=' * 60)
for s in result['sources']:
    print(f"  [{s['source']} — chunk {s['chunk_id']}]")

In [ ]:
# Multi-turn example — pass chat history to maintain context
history = [
    {'role': 'user', 'content': 'What was the revenue growth YoY?'},
    {'role': 'assistant', 'content': result['answer']}
]

followup = agent.run(
    query='Which segment contributed the most to that growth?',
    chat_history=history
)

print(followup['answer'])

## 5. Quick Retrieval Evaluation

A minimal eval to check that relevant chunks are actually being surfaced for a set of known questions.
For production, this can be extended with RAGAS or a custom scoring framework.

In [ ]:
eval_set = [
    {'query': 'What was the net income this quarter?',         'expected_keyword': 'net income'},
    {'query': 'What are the EPS figures?',                    'expected_keyword': 'earnings per share'},
    {'query': 'What is the ARR for the cloud segment?',       'expected_keyword': 'Annual Recurring Revenue'},
    {'query': 'What acquisition was announced?',              'expected_keyword': 'DataStream'},
    {'query': 'What did the CFO say about free cash flow?',   'expected_keyword': 'free cash flow'},
]

hits = 0
for item in eval_set:
    results = vs.search(item['query'], top_k=4)
    combined_text = ' '.join(r['content'] for r in results).lower()
    found = item['expected_keyword'].lower() in combined_text
    hits += int(found)
    status = '[PASS]' if found else '[FAIL]'
    print(f"{status}  {item['query']}")

print(f'\nRetrieval hit rate: {hits}/{len(eval_set)} ({100*hits/len(eval_set):.0f}%)')